# 02 · External data: Census ACS by ZIP (ZCTA)

Pulls neighborhood-level demographics for every ZIP Code Tabulation Area (ZCTA) in the 8 states in the data (AL, CT, FL, GA, MN, NJ, NY, WI). They're joined to policies on `ZIP_CD` in the feature step.

**Source:** the American Community Survey **2018 5-year** estimates (covering 2014–2018), from the Census API. This vintage is the one closest to the 2015–2018 quote dates without using later data. The API key is read from `CENSUS_API_KEY` in `.env`, which git ignores.

| Table | Content | Derived features |
|---|---|---|
| B19013 | Median household income | `ACS_MED_HH_INCOME` |
| B01003 | Total population | `ACS_POPULATION`, `ACS_POP_DENSITY` (÷ Gazetteer land area) |
| B01002 | Median age | `ACS_MED_AGE` |
| B25003 | Tenure (owner vs renter) | `ACS_OWNER_OCC_SHARE` |
| B08201 | Vehicles available per household | `ACS_VEH_PER_HH`, `ACS_NO_VEH_SHARE` |
| B08301 | Means of transportation to work | `ACS_COMMUTE_{DROVE_ALONE, CARPOOL, TRANSIT, WALK_BIKE, WFH}` (share of workers) |
| B08303 | Travel time to work | `ACS_COMMUTE_MEAN_MIN`, `ACS_COMMUTE_45PLUS` |
| B15003 | Educational attainment (age 25+) | `ACS_EDU_NO_HS`, `ACS_EDU_BACHELOR_PLUS` |

Land area comes from the **2018 Census Gazetteer** ZCTA file (`ALAND_SQMI`).

**Outputs** (`data/external/`):
- `acs2018_<TABLE>.parquet`: raw estimate columns per table, cached so re-runs skip the API.
- `gazetteer2018_zcta.parquet`: the land-area file.
- `acs2018_zcta_features.parquet`: one row per ZCTA with the derived features above.

In [1]:
import io, os, zipfile
from pathlib import Path
import numpy as np, pandas as pd, requests
from dotenv import load_dotenv

load_dotenv('../.env')
KEY = os.environ['CENSUS_API_KEY']
API = 'https://api.census.gov/data/2018/acs/acs5'
GAZ_URL = 'https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2018_Gazetteer/2018_Gaz_zcta_national.zip'
STATES = {'AL': '01', 'CT': '09', 'FL': '12', 'GA': '13', 'MN': '27', 'NJ': '34', 'NY': '36', 'WI': '55'}  # state FIPS codes
TABLES = ['B19013', 'B01003', 'B01002', 'B25003', 'B08201', 'B08301', 'B08303', 'B15003']
EXT = Path('../data/external'); EXT.mkdir(parents=True, exist_ok=True)

pol = pd.read_parquet('../data/processed/policies.parquet', columns=['STATE_CD', 'ZIP_CD'])
assert set(pol.STATE_CD) == set(STATES)

## 1. Download ACS tables
- `group(<TABLE>)` returns every variable in a table in a single call.
- For 2018 the API only serves ZCTAs **nested within a state** (asking for one ZCTA directly fails as "ambiguous geography"), so each table is requested once per state.
- Only estimate columns (`_###E`) are kept; margins of error (`M`) and annotation flags (`EA`/`MA`) are dropped.
- The Census encodes "not available" as large negative sentinels (e.g. `-666666666` when a ZCTA is too small for a median), which are converted to NaN.
- A ZCTA that crosses a state line is returned once per state with identical values (checked below), so duplicates are dropped.

In [2]:
def fetch(table):
    frames = []
    for fips in STATES.values():
        r = requests.get(API, params={'get': f'group({table})', 'for': 'zip code tabulation area:*', 'in': f'state:{fips}', 'key': KEY}, timeout=120)
        r.raise_for_status()
        frames.append(pd.DataFrame(r.json()[1:], columns=r.json()[0]))
    df = pd.concat(frames).rename(columns={'zip code tabulation area': 'ZCTA'})
    est = [c for c in df if c.startswith(table) and c.endswith('E')]
    df[est] = df[est].apply(pd.to_numeric).mask(lambda d: d < 0)
    assert df.groupby('ZCTA')[est].nunique(dropna=False).le(1).all().all()  # cross-state duplicates are identical
    return df[['ZCTA', *est]].drop_duplicates('ZCTA').set_index('ZCTA')

acs = {}
for t in TABLES:
    f = EXT / f'acs2018_{t}.parquet'
    if not f.exists():
        fetch(t).to_parquet(f)
    acs[t] = pd.read_parquet(f)
pd.DataFrame({t: [len(d), d.shape[1], d.isna().any(axis=1).sum()] for t, d in acs.items()}, index=['ZCTAs', 'estimate cols', 'ZCTAs w/ NaN'])

,B19013,B01003,B01002,B25003,B08201,B08301,B08303,B15003
ZCTAs,6690,6690,6690,6690,6690,6690,6690,6690
estimate cols,1,1,3,3,30,21,13,25
ZCTAs w/ NaN,355,0,197,0,0,0,0,0


## 2. Land area (Gazetteer)
A tab-separated national file with one row per ZCTA. Its last column name comes with trailing whitespace, so names are stripped. `GEOID` is read as a string so leading zeros survive.

In [3]:
f = EXT / 'gazetteer2018_zcta.parquet'
if not f.exists():
    z = zipfile.ZipFile(io.BytesIO(requests.get(GAZ_URL, timeout=120).content))
    g = pd.read_csv(z.open(z.namelist()[0]), sep='\t', dtype={'GEOID': str})
    g.columns = g.columns.str.strip()
    g.rename(columns={'GEOID': 'ZCTA'}).set_index('ZCTA').to_parquet(f)
gaz = pd.read_parquet(f)
gaz.head(3)

,ALAND,AWATER,ALAND_SQMI,AWATER_SQMI,INTPTLAT,INTPTLONG
ZCTA,,,,,,
00601,166659749,799292,64.348,0.309,18.180555,-66.749961
00602,79307535,4428429,30.621,1.710,18.361945,-67.175597
00603,81887185,181411,31.617,0.070,18.455183,-67.119887


## 3. Derive features
Counts are converted to **shares** so that ZCTAs of different sizes are comparable.
- `ACS_VEH_PER_HH`: households with "4 or more" vehicles are counted as 4, which slightly underestimates the true mean.
- `ACS_COMMUTE_*` mode shares use all workers aged 16+ as the denominator, so `WFH` is a share of workers.
- `ACS_COMMUTE_MEAN_MIN` is estimated from bin midpoints (90+ min → 100). The denominator is workers who did *not* work from home, because B08303 excludes them.
- `ACS_EDU_NO_HS` covers everything below a high school diploma or GED. `ACS_EDU_BACHELOR_PLUS` covers bachelor's, master's, professional and doctorate degrees.
- ZCTAs with zero population give 0/0 → NaN, and zero land area gives density ∞ → NaN.
- `ACS_MED_HH_INCOME` is top-coded by the Census: `250001` means "$250,000 or more". It's left as-is, which is harmless for tree models.

In [4]:
a, cols = acs, lambda t, lo, hi: [f'{t}_{i:03d}E' for i in range(lo, hi + 1)]
hh, cm, tt, ed = a['B08201'], a['B08301'], a['B08303'], a['B15003']
pop, land = a['B01003'].B01003_001E, gaz.ALAND_SQMI.reindex(a['B01003'].index)
commute_mid = [2.5, 7, 12, 17, 22, 27, 32, 37, 42, 52, 74.5, 100]  # minutes, bins B08303_002..013

feat = pd.DataFrame({
    'ACS_MED_HH_INCOME':       a['B19013'].B19013_001E,
    'ACS_POPULATION':          pop,
    'ACS_LAND_SQMI':           land,
    'ACS_POP_DENSITY':         pop / land,
    'ACS_MED_AGE':             a['B01002'].B01002_001E,
    'ACS_OWNER_OCC_SHARE':     a['B25003'].B25003_002E / a['B25003'].B25003_001E,
    'ACS_VEH_PER_HH':          hh[cols('B08201', 3, 6)] @ [1, 2, 3, 4] / hh.B08201_001E,
    'ACS_NO_VEH_SHARE':        hh.B08201_002E / hh.B08201_001E,
    'ACS_COMMUTE_DROVE_ALONE': cm.B08301_003E / cm.B08301_001E,
    'ACS_COMMUTE_CARPOOL':     cm.B08301_004E / cm.B08301_001E,
    'ACS_COMMUTE_TRANSIT':     cm.B08301_010E / cm.B08301_001E,
    'ACS_COMMUTE_WALK_BIKE':   (cm.B08301_018E + cm.B08301_019E) / cm.B08301_001E,
    'ACS_COMMUTE_WFH':         cm.B08301_021E / cm.B08301_001E,
    'ACS_COMMUTE_MEAN_MIN':    tt[cols('B08303', 2, 13)] @ commute_mid / tt.B08303_001E,
    'ACS_COMMUTE_45PLUS':      tt[cols('B08303', 11, 13)].sum(axis=1) / tt.B08303_001E,
    'ACS_EDU_NO_HS':           ed[cols('B15003', 2, 16)].sum(axis=1) / ed.B15003_001E,
    'ACS_EDU_BACHELOR_PLUS':   ed[cols('B15003', 22, 25)].sum(axis=1) / ed.B15003_001E,
}).replace([np.inf, -np.inf], np.nan).rename_axis('ZCTA')
feat.to_parquet(EXT / 'acs2018_zcta_features.parquet')
feat.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
ACS_MED_HH_INCOME,6335.0,64278.09,28654.21,9856.0,46062.50,57500.00,75215.50,250001.00
ACS_POPULATION,6690.0,11830.52,15383.30,0.0,1318.00,4925.50,17396.25,112425.00
ACS_LAND_SQMI,6690.0,51.01,72.12,0.0,5.72,26.30,68.76,1118.67
ACS_POP_DENSITY,6690.0,2501.59,9529.98,0.0,36.51,169.89,1483.48,152693.29
ACS_MED_AGE,6582.0,42.89,8.40,2.4,37.70,42.40,47.10,87.80
ACS_OWNER_OCC_SHARE,6564.0,0.73,0.19,0.0,0.66,0.77,0.85,1.00
ACS_VEH_PER_HH,6564.0,1.85,0.39,0.0,1.67,1.90,2.08,4.00
ACS_NO_VEH_SHARE,6564.0,0.07,0.11,0.0,0.02,0.05,0.08,1.00
ACS_COMMUTE_DROVE_ALONE,6577.0,0.78,0.15,0.0,0.75,0.81,0.85,1.00
ACS_COMMUTE_CARPOOL,6577.0,0.08,0.06,0.0,0.05,0.08,0.11,1.00


## 4. Coverage vs. policy ZIPs
ZIP codes and ZCTAs are not identical: PO-box-only and single-business ZIPs have no ZCTA. ZCTAs with tiny populations can also lack medians. **Result:** all 1,168 distinct policy ZIPs match a ZCTA. About 4% of policies sit in a ZCTA without a median income; these stay NaN, or the feature step can fill them from `ZIP3` population-weighted averages.

In [5]:
zips = pol.ZIP_CD.dropna()
print(f'policies with a ZIP that matches a ZCTA:     {zips.isin(feat.index).mean():.1%}')
print(f'policies whose ZCTA has median income:       {zips.map(feat.ACS_MED_HH_INCOME).notna().mean():.1%}')
print(f'distinct policy ZIPs: {zips.nunique()}, unmatched: {(~pd.Index(zips.unique()).isin(feat.index)).sum()}')

policies with a ZIP that matches a ZCTA:     100.0%
policies whose ZCTA has median income:       96.1%
distinct policy ZIPs: 1168, unmatched: 0
